# LlamaParse — Detector + Classificador + Qualidade Textual

**LlamaParse atua em três frentes:**

1. **Detector (Parte A)** — processa cada PDF integralmente, gera markdown com figuras, tabelas e equações estruturadas.
2. **Classificador (Parte B)** — recebe cada crop produzido pelos detectores e classifica o tipo de gráfico via prompt customizado JSON.
3. **Comparação textual (Parte C)** — texto extraído comparado lado a lado com Chandra OCR 2.

**Configuração:**

- API LlamaParse acessada via SDK Python (`llama_cloud_services`).
- Modo `parse_page_with_lvm` (≈ "accurate" — usa large vision model).
- Custo aproximado: 3 créditos/página. Plano gratuito tem 1000 créditos/dia.
- Chave da API armazenada como Colab Secret `LLAMA_CLOUD_API_KEY` (nunca hardcoded).

**Detalhes importantes:**

- Usa `aload_data` (async) ao invés de `load_data` para evitar erro `Event loop is closed` em chamadas sequenciais.
- Detecção de quota — se a API retornar `exceeded credits`, o loop para imediatamente.
- Retomada inteligente — descarta linhas com erro e reprocessa só o que faltou.


## 1. Instalação

In [ ]:
!pip install -q llama-parse llama-cloud-services pymupdf pillow pandas nest_asyncio

In [ ]:
# Carrega chave do Colab Secrets (NUNCA hardcoded)
import os
from google.colab import userdata
try:
    api_key = userdata.get('LLAMA_CLOUD_API_KEY')
    os.environ['LLAMA_CLOUD_API_KEY'] = api_key
    print(f"✓ Chave carregada: {api_key[:10]}...{api_key[-4:]}  ({len(api_key)} chars)")
except Exception as e:
    print(f"✗ Erro: {e}")
    print("→ Configure em Colab → 🔑 Secrets → LLAMA_CLOUD_API_KEY")

## 2. Configuração de pastas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

INPUT_DIR     = Path('/content/drive/MyDrive/Chandra2/PDF')
OUTPUT_ROOT   = Path('/content/drive/MyDrive/Chandra2/output')
CHANDRA_DIR   = OUTPUT_ROOT / 'chandra'
RASTER_DIR    = OUTPUT_ROOT / 'pymupdf_raster'
LLAMA_DIR     = OUTPUT_ROOT / 'llamaparse'
GT_CSV        = OUTPUT_ROOT / 'avaliacao_manual.csv'

LLAMA_DIR.mkdir(parents=True, exist_ok=True)

# 10 PDFs alvo (mesmos do N2)
TARGET_NAMES = [
    'W3132421076.pdf',
    'Mattos(2018)-IA 163 - Artigo Citros-Cafe - Fernanda Bochi Dos Santos.pdf',
    'W3110745114.pdf',
    'W4309496579.pdf',
    'Yamane(2022)-horticulturae-08-01126 - Janaina Lais Pacheco Lara Morandin (1).pdf',
    'W3138442591.pdf',
    'grafico de radar.pdf',
    'W4323846702.pdf',
    'Mattos(2018)-TodaFruta2 - Fernanda Bochi Dos Santos.pdf',
    'W3216639369.pdf',
]
pdfs = [INPUT_DIR / n for n in TARGET_NAMES if (INPUT_DIR / n).exists()]
print(f"PDFs encontrados: {len(pdfs)}/{len(TARGET_NAMES)}")

## 3. PARTE A — LlamaParse como Detector (camada 1)

Processa cada PDF inteiro, salva markdown e imagens em `output/llamaparse/<pdf_stem>/`.

In [ ]:
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type="markdown",
    parse_mode="parse_page_with_lvm",  # modo "accurate", usa VLM
    extract_charts=True,
    verbose=False,
    num_workers=4,
)
print("✓ Parser criado")

In [ ]:
# Processa os 10 PDFs (sequencial pra log mais claro)
import time, json

for pdf in pdfs:
    out_dir = LLAMA_DIR / pdf.stem
    md_path = out_dir / 'output.md'
    if md_path.exists() and md_path.stat().st_size > 100:
        print(f"  ⏭  {pdf.name} já processado, pulando")
        continue

    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n→ {pdf.name}")
    t0 = time.time()
    try:
        docs = parser.load_data(str(pdf))
        text = "\n\n".join(d.text for d in docs)
        md_path.write_text(text, encoding='utf-8')

        # Salva também json bruto das páginas
        pages_raw = [{'page': i+1, 'text': d.text, 'metadata': dict(d.metadata)}
                     for i, d in enumerate(docs)]
        (out_dir / 'raw_pages.json').write_text(
            json.dumps(pages_raw, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print(f"  ✓ {len(docs)} páginas, {len(text)} chars  ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  ✗ ERRO: {e}")

print("\n=== Parte A concluída ===")

In [ ]:
# Conta elementos no markdown (mesmo método dos outros notebooks)
import re
from collections import Counter

def count_in_md(md_text):
    n_images = len(re.findall(r'!\[[^\]]*\]\([^)]+\)', md_text))
    n_tables = len(re.findall(r'<table[^>]*>', md_text, flags=re.IGNORECASE))
    md_table_blocks = re.findall(r'(?:^\|[^\n]+\|\n){2,}', md_text, flags=re.MULTILINE)
    n_tables += len(md_table_blocks)
    n_equations = len(re.findall(r'\$\$[^$]+\$\$', md_text))
    return {'image': n_images, 'table': n_tables, 'equation': n_equations}

llama_results = {}
for pdf in pdfs:
    md_path = LLAMA_DIR / pdf.stem / 'output.md'
    if md_path.exists():
        text = md_path.read_text(encoding='utf-8')
        llama_results[pdf.name] = count_in_md(text)
    else:
        llama_results[pdf.name] = {'image': 0, 'table': 0, 'equation': 0}

print(f"{'PDF':<55} {'img':>4} {'tab':>4} {'eq':>4}")
for name, c in llama_results.items():
    print(f"  {name[:55]:<55} {c['image']:>4} {c['table']:>4} {c['equation']:>4}")

## 4. Comparação dos detectores nos 10 PDFs

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# Carrega contagens dos outros detectores

# ----- Chandra -----
def chandra_counts(pdf_stem):
    sub = CHANDRA_DIR / pdf_stem
    html_p = sub / f"{pdf_stem}.html"
    if not html_p.exists():
        return {'image': 0, 'table': 0}
    soup = BeautifulSoup(html_p.read_text(encoding='utf-8'), 'html.parser')
    return {'image': len(soup.find_all('img')),
            'table': len(soup.find_all('table'))}

# ----- PyMuPDF raster -----
def raster_count(pdf_stem):
    sub = OUTPUT_ROOT / 'pymupdf_raster' / pdf_stem
    if not sub.exists(): return 0
    return len(list(sub.glob('*.png')))

# ----- Constrói tabela bruta -----
rows = []
for pdf in pdfs:
    ch = chandra_counts(pdf.stem)
    ll = llama_results[pdf.name]
    rows.append({
        'pdf': pdf.name[:35],
        'Chandra img': ch['image'],
        'Chandra tab': ch['table'],
        'PyMuPDF rast': raster_count(pdf.stem),
        'Llama img':  ll['image'],
        'Llama tab':  ll['table'],
    })
df_det = pd.DataFrame(rows)
print(df_det.to_string(index=False))

df_det.to_csv(LLAMA_DIR / 'comparacao_detectores.csv', index=False)
print(f"\nSalvo: {LLAMA_DIR/'comparacao_detectores.csv'}")

In [ ]:
# Ranking com GT
import numpy as np

gt_df = pd.read_csv(GT_CSV).set_index('pdf')
common = [p.name for p in pdfs if p.name in gt_df.index]
print(f"Avaliações encontradas: {len(common)}/{len(pdfs)}\n")

gt_graf_img = [int(gt_df.loc[n, 'gt_graficos']) + int(gt_df.loc[n, 'gt_imagens']) for n in common]
gt_tab      = [int(gt_df.loc[n, 'gt_tabelas']) for n in common]

chand_img = [chandra_counts(Path(n).stem)['image'] for n in common]
chand_tab = [chandra_counts(Path(n).stem)['table'] for n in common]
rast_img  = [raster_count(Path(n).stem) for n in common]
llama_img = [llama_results[n]['image'] for n in common]
llama_tab = [llama_results[n]['table'] for n in common]

def metrics(detected, gt):
    d, g = np.array(detected), np.array(gt)
    err = d - g
    return {
        'detectado': int(d.sum()), 'GT': int(g.sum()),
        'MAE':  round(float(np.abs(err).mean()), 2),
        'viés': round(float(err.mean()), 2),
        'perfeitos': int((err == 0).sum()),
    }

results = {
    'Chandra (img/fig)':   metrics(chand_img, gt_graf_img),
    'Chandra (table)':     metrics(chand_tab, gt_tab),
    'PyMuPDF raster':      metrics(rast_img, gt_graf_img),
    'LlamaParse (img)':    metrics(llama_img, gt_graf_img),
    'LlamaParse (table)':  metrics(llama_tab, gt_tab),
}

ranking = pd.DataFrame(results).T.sort_values('MAE')
print("=== Ranking de detecção (menor MAE = melhor) ===\n")
print(ranking.to_string())

ranking.to_csv(LLAMA_DIR / 'ranking_com_llamaparse.csv')
print(f"\nSalvo: {LLAMA_DIR/'ranking_com_llamaparse.csv'}")

## 5. PARTE B — LlamaParse como Classificador (camada 2)

Para cada crop produzido pelo Chandra ou PyMuPDF raster, empacota como PDF de 1 página e envia ao LlamaParse com prompt JSON customizado pedindo `chart_type`, `title`, `data_csv`, `description`.

**Importante:** usa `aload_data` (async) ao invés de `load_data` (sync) — a versão sync alterna chamadas com erro `Event loop is closed` em ambiente Colab/Jupyter.

In [ ]:
# ----- Coleta crops dos detectores -----
all_crops = []

# Chandra
for sub in CHANDRA_DIR.iterdir():
    if not sub.is_dir(): continue
    for crop in sub.glob('*.webp'):
        all_crops.append({'pdf': sub.name, 'detector': 'chandra', 'crop_path': str(crop)})
    for crop in sub.glob('*.png'):
        all_crops.append({'pdf': sub.name, 'detector': 'chandra', 'crop_path': str(crop)})

# PyMuPDF raster
for sub in RASTER_DIR.iterdir():
    if not sub.is_dir(): continue
    for crop in sub.glob('*.png'):
        all_crops.append({'pdf': sub.name, 'detector': 'pymupdf_raster', 'crop_path': str(crop)})

from collections import Counter
print(f"Total crops: {len(all_crops)}")
print(dict(Counter(c['detector'] for c in all_crops)))

In [ ]:
# Helper: empacota cada crop como PDF de 1 página (LlamaParse processa PDFs, não imagens)
import fitz
from PIL import Image
from pathlib import Path

def crop_to_pdf(img_path, out_pdf_path):
    img = Image.open(img_path).convert('RGB')
    doc = fitz.open()
    rect = fitz.Rect(0, 0, img.width, img.height)
    page = doc.new_page(width=img.width, height=img.height)
    img_bytes = Path('/tmp/_tmp_img.png')
    img.save(img_bytes)
    page.insert_image(rect, filename=str(img_bytes))
    doc.save(str(out_pdf_path))
    doc.close()

In [ ]:
# Parser dedicado pra classificação (com user_prompt JSON estruturado)
CLASSIFY_PROMPT = '''You are looking at a single chart/figure cropped from a scientific paper.

Identify and return in JSON format:
- "chart_type": one of [bar, line, multi_line, multi_bar, pie, scatter, radar, histogram, multi_histogram, heatmap, boxplot, area, table, photo, diagram, logo, text_fragment, not_a_chart]
- "title": the chart title if visible, else null
- "data_csv": a CSV representation of the chart data (only filled if you can extract numeric data)
- "description": a 1-2 sentence description

Return ONLY valid JSON.'''

classifier = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type="markdown",
    parse_mode="parse_page_with_lvm",
    user_prompt=CLASSIFY_PROMPT,
    verbose=False,
    num_workers=2,
)
print("✓ Classifier criado")

In [ ]:
# Inferência completa com loop async + detecção de quota
import asyncio, time, re, json as _json, shutil
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

TMP_DIR = Path('/tmp/llama_crops')
TMP_DIR.mkdir(exist_ok=True)
OUT_CSV = LLAMA_DIR / 'llamaparse_classifications.csv'

# Retomada inteligente
done = set()
rows = []
if OUT_CSV.exists():
    backup = Path(str(OUT_CSV).replace('.csv', '_BACKUP_anterior.csv'))
    shutil.copy(OUT_CSV, backup)
    existing = pd.read_csv(OUT_CSV)
    existing['has_error'] = (
        existing['raw_response'].astype(str).str.contains(
            'ERROR:|exceeded|credit|Event loop', case=False, na=False)
        | (existing['tipo_predito'].astype(str).str.strip() == '')
        | (existing['tipo_predito'].astype(str).str.lower() == 'nan')
    )
    valid = existing[~existing['has_error']].drop(columns=['has_error'])
    done = set(valid['crop_path'].tolist())
    rows = valid.to_dict('records')
    print(f"📂 Aproveitando {len(done)} crops válidos do CSV anterior")
    print(f"   ({len(existing) - len(done)} a reprocessar)")

to_process = [c for c in all_crops if c['crop_path'] not in done]
print(f"\nA processar: {len(to_process)} crops\n")

QUOTA_KEYWORDS = ['quota', 'rate limit', 'insufficient', 'credit', '402', '429',
                  'no credits', 'exceeded', 'subscription', 'maximum number']
def is_quota_error(text):
    return any(kw in str(text).lower() for kw in QUOTA_KEYWORDS)

async def process_all():
    quota_hit = False
    t0_glob = time.time()
    for i, item in enumerate(to_process, 1):
        if quota_hit: break

        crop_pdf = TMP_DIR / f"crop_{i:04d}.pdf"
        raw_text = ''
        try:
            crop_to_pdf(item['crop_path'], crop_pdf)
            docs = await classifier.aload_data(str(crop_pdf))  # ASYNC
            raw_text = "\n".join(d.text for d in docs)
            if is_quota_error(raw_text):
                print(f"\n⚠️ QUOTA detectado: {raw_text[:150]}")
                quota_hit = True
                continue
        except Exception as e:
            raw_text = f"ERROR: {e}"
            if is_quota_error(e):
                print(f"\n⚠️ QUOTA: {e}")
                quota_hit = True
                continue

        parsed = {'chart_type': '', 'title': '', 'data_csv': '', 'description': ''}
        if not raw_text.startswith('ERROR:'):
            try:
                m = re.search(r'\{[\s\S]*\}', raw_text)
                if m:
                    j = _json.loads(m.group(0))
                    for k in parsed.keys():
                        parsed[k] = str(j.get(k, '') or '')[:3000]
            except Exception: pass

        rows.append({
            'pdf':             item['pdf'],
            'detector':        item['detector'],
            'crop_path':       item['crop_path'],
            'crop_filename':   Path(item['crop_path']).name,
            'tipo_predito':    parsed['chart_type'],
            'titulo':          parsed['title'],
            'csv_dados':       parsed['data_csv'],
            'descricao':       parsed['description'],
            'raw_response':    raw_text[:800],
            'tipo_real':       '',
            'classificou_certo': '',
        })

        if i % 5 == 0 or i == len(to_process):
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
            elapsed = (time.time() - t0_glob) / 60
            print(f"  [{i}/{len(to_process)}] checkpoint ({elapsed:.1f} min)")

        status = parsed['chart_type'][:35] if parsed['chart_type'] else 'sem JSON'
        print(f"     {item['detector']:<15} {Path(item['crop_path']).name[:28]:<30} → {status}")

    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    print(f"\n=== Concluído em {(time.time()-t0_glob)/60:.1f} min ===")
    print(f"   Total no CSV: {len(rows)} crops")
    if quota_hit:
        print(f"\n⚠️ Loop interrompido por quota — renove créditos e rode de novo")

asyncio.run(process_all())

## 6. PARTE C — Comparação textual (Chandra vs LlamaParse)

In [ ]:
# Estatísticas básicas: tamanho do markdown, número de palavras, contagem de elementos
import pandas as pd

def stats_md(text):
    if not text: return {'chars': 0, 'words': 0}
    return {'chars': len(text), 'words': len(text.split())}

rows_qual = []
for pdf in pdfs:
    ch_html = CHANDRA_DIR / pdf.stem / f"{pdf.stem}.html"
    ch_text = ""
    if ch_html.exists():
        soup = BeautifulSoup(ch_html.read_text(encoding='utf-8'), 'html.parser')
        ch_text = soup.get_text(' ', strip=True)

    ll_md = LLAMA_DIR / pdf.stem / 'output.md'
    ll_text = ll_md.read_text(encoding='utf-8') if ll_md.exists() else ""

    ch_s = stats_md(ch_text)
    ll_s = stats_md(ll_text)
    rows_qual.append({
        'pdf': pdf.name[:35],
        'Chandra_chars': ch_s['chars'], 'Chandra_words': ch_s['words'],
        'Llama_chars':   ll_s['chars'], 'Llama_words':   ll_s['words'],
        'ratio_words':   round(ll_s['words'] / ch_s['words'], 2) if ch_s['words'] else 0,
    })

df_qual = pd.DataFrame(rows_qual)
print(df_qual.to_string(index=False))
df_qual.to_csv(LLAMA_DIR / 'qualidade_textual.csv', index=False)
print(f"\nSalvo: {LLAMA_DIR/'qualidade_textual.csv'}")

In [ ]:
# Trecho lado-a-lado de UM PDF para inspeção qualitativa
target_pdf = pdfs[0]
ch_html = CHANDRA_DIR / target_pdf.stem / f"{target_pdf.stem}.html"
ll_md   = LLAMA_DIR / target_pdf.stem / 'output.md'

if ch_html.exists() and ll_md.exists():
    soup = BeautifulSoup(ch_html.read_text(encoding='utf-8'), 'html.parser')
    ch_text = soup.get_text('\n', strip=True)[:500]
    ll_text = ll_md.read_text(encoding='utf-8')[:500]
    print(f"=== {target_pdf.name} ===\n")
    print("--- CHANDRA (primeiros 500 chars) ---")
    print(ch_text)
    print("\n--- LLAMAPARSE (primeiros 500 chars) ---")
    print(ll_text)
else:
    print("Arquivos não encontrados pra comparar")

## 7. Métricas finais — qual classificador foi melhor?

Roda **depois** que você preencher `classificou_certo` e `tipo_real` em uma quantidade significativa de crops via inspeção manual.

In [ ]:
import pandas as pd

df = pd.read_csv(OUT_CSV)

# Versão tolerante a NaN
def is_valid_eval(v):
    if pd.isna(v): return False
    return str(v).lower().strip() in {'sim','nao','parcial'}

avaliadas = df[df['classificou_certo'].apply(is_valid_eval)].copy()
print(f"Crops avaliados: {len(avaliadas)} / {len(df)}")

if len(avaliadas) == 0:
    print("\n⚠️ Nenhum crop avaliado ainda. Preencha o CSV primeiro.")
else:
    score_map = {'sim': 1.0, 'parcial': 0.5, 'nao': 0.0}
    avaliadas['score'] = (avaliadas['classificou_certo']
                          .astype(str).str.lower().str.strip().map(score_map))

    by_det = avaliadas.groupby('detector').agg(
        n=('score','count'),
        n_acerto=('score', lambda s: (s==1.0).sum()),
        n_parcial=('score', lambda s: (s==0.5).sum()),
        n_erro=('score', lambda s: (s==0.0).sum()),
        acerto_strict=('score', lambda s: (s==1.0).mean()),
        acerto_lenient=('score','mean'),
    ).round(3).sort_values('acerto_strict', ascending=False)
    print("\n=== Classificação LlamaParse — acurácia por DETECTOR de origem ===\n")
    print(by_det.to_string())

    by_det.to_csv(LLAMA_DIR / 'acuracia_classificacao_llamaparse.csv')
    winner = by_det['acerto_strict'].idxmax()
    print(f"\n🏆 Detector mais classificável pelo LlamaParse:")
    print(f"   {winner} — strict {by_det.loc[winner,'acerto_strict']*100:.1f}%, "
          f"lenient {by_det.loc[winner,'acerto_lenient']*100:.1f}%")

## 8. Conclusão

**Resultados observados (sprint 29/abr–13/mai 2026):**

- **Parte A (Detector):** LlamaParse atinge MAE 6,3 em tabelas com viés de apenas +2,3 (vs Chandra MAE 5,4 mas viés +4,6). Em figuras, Chandra é melhor (MAE 5,5 vs LlamaParse 8,7).
- **Parte B (Classificador):** atinge **91,7% lenient** sobre crops do Chandra e **91,3%** sobre PyMuPDF raster — atendendo o critério de 90% definido pela sprint.
- **Parte C (Qualidade textual):** volume de texto extraído é equivalente ao Chandra (razão ≈ 0,96), mas com estilo mais coloquial e menos preservação de estrutura.

**Considerações de produção:**

- LlamaParse exige plano pago para uso intensivo (1000 créditos/dia no free; cada crop consome ~3 créditos).
- A versão async (`aload_data`) é obrigatória em Colab para evitar `Event loop is closed` em chamadas sequenciais.
- A chave da API deve estar em Colab Secrets, nunca hardcoded no notebook.
